## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy import stats

In [9]:
df = pd.read_csv("/content/gdrive/MyDrive/STAT3013/Dataset/data_cleaned.csv")
df["volume_category"] = pd.Categorical(df["volume_category"],
    categories=["Low","Moderate","High"], ordered=True)

ALPHA = 0.05
print(f"Loaded: {df.shape[0]} rows")
print(f"Alpha = {ALPHA}")

Loaded: 198 rows
Alpha = 0.05


## 1. t-test: Low vs High Volume (hedges_g)
**H0:** Mean hedges_g của Low = High
**H1:** Khác nhau có ý nghĩa thống kê

In [10]:
low  = df[df["volume_category"] == "Low"]["hedges_g"].dropna()
high = df[df["volume_category"] == "High"]["hedges_g"].dropna()

print(f"Low:  n={len(low)},  mean={low.mean():.4f}, SD={low.std():.4f}")
print(f"High: n={len(high)}, mean={high.mean():.4f}, SD={high.std():.4f}")

Low:  n=48,  mean=0.3051, SD=0.3871
High: n=89, mean=0.5416, SD=0.3803


In [11]:
# Levene's test
print("\n--- Levene\'s Test (equality of variances) ---")
print(stats.levene(low, high, center="median"))


--- Levene's Test (equality of variances) ---
LeveneResult(statistic=np.float64(0.09876327435432952), pvalue=np.float64(0.7538039677922802))


In [12]:
# Welch two-sample t-test
print("\n--- Welch t-test (Low vs High Volume) ---")
print(stats.ttest_ind(low, high, equal_var=False))


--- Welch t-test (Low vs High Volume) ---
TtestResult(statistic=np.float64(-3.431866371515042), pvalue=np.float64(0.0008894621523165667), df=np.float64(94.92155110250833))


## 2. One-way ANOVA: Low / Moderate / High Volume
**H0:** Mean hedges_g bằng nhau ở 3 nhóm
**H1:** Ít nhất 1 nhóm khác biệt

In [13]:
g_low = df[df["volume_category"] == "Low"]["hedges_g"].dropna()
g_mod = df[df["volume_category"] == "Moderate"]["hedges_g"].dropna()
g_high= df[df["volume_category"] == "High"]["hedges_g"].dropna()

print(f"Low:      n={len(g_low)},  mean={g_low.mean():.4f}")
print(f"Moderate: n={len(g_mod)},  mean={g_mod.mean():.4f}")
print(f"High:     n={len(g_high)}, mean={g_high.mean():.4f}")

print("\n--- One-way ANOVA ---")
print(stats.f_oneway(g_low, g_mod, g_high))

Low:      n=48,  mean=0.3051
Moderate: n=61,  mean=0.3624
High:     n=89, mean=0.5416

--- One-way ANOVA ---
F_onewayResult(statistic=np.float64(8.392016271648464), pvalue=np.float64(0.0003189967833903542))


## 3. Tukey HSD Post-hoc
Xác định cặp nhóm nào khác biệt có ý nghĩa sau ANOVA

In [14]:
# statsmodels Tukey HSD
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=df["hedges_g"].dropna(),
    groups=df.loc[df["hedges_g"].notna(), "volume_category"],
    alpha=ALPHA
)
print(tukey.summary())

 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1  group2  meandiff p-adj   lower   upper  reject
------------------------------------------------------
  High      Low  -0.2365 0.0008 -0.3873 -0.0856   True
  High Moderate  -0.1791  0.008 -0.3191 -0.0391   True
   Low Moderate   0.0573 0.6829 -0.1052  0.2198  False
------------------------------------------------------


## 4. Chi-square: failure_binary vs volume_category
**H0:** failure_binary và volume_category độc lập

In [15]:
ct = pd.crosstab(df["volume_category"], df["failure_binary"],
                rownames=["Volume Category"],
                colnames=["Failure Binary"])
print("Contingency Table:")
print(ct)

print("\n--- Chi-square Test ---")
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(f"Chi2={chi2:.3f}, p={p:.4f}, dof={dof}")

Contingency Table:
Failure Binary    0   1
Volume Category        
Low              11  37
Moderate          9  52
High             19  70

--- Chi-square Test ---
Chi2=1.410, p=0.4940, dof=2


## 5. t-test: Trained vs Untrained
**H0:** Mean hedges_g bằng nhau giữa 2 nhóm

In [16]:
trained   = df[df["train.status"] == "trained"]["hedges_g"].dropna()
untrained = df[df["train.status"] == "untrained"]["hedges_g"].dropna()

print(f"Trained:   n={len(trained)},  mean={trained.mean():.4f}, SD={trained.std():.4f}")
print(f"Untrained: n={len(untrained)}, mean={untrained.mean():.4f}, SD={untrained.std():.4f}")

print("\n--- Levene\'s Test ---")
print(stats.levene(trained, untrained, center="median"))

print("\n--- Welch t-test (Trained vs Untrained) ---")
print(stats.ttest_ind(trained, untrained, equal_var=False))

Trained:   n=117,  mean=0.4660, SD=0.3427
Untrained: n=81, mean=0.3757, SD=0.4019

--- Levene's Test ---
LeveneResult(statistic=np.float64(0.11774333435303615), pvalue=np.float64(0.7318627883070897))

--- Welch t-test (Trained vs Untrained) ---
TtestResult(statistic=np.float64(1.648196208687475), pvalue=np.float64(0.10135201560372156), df=np.float64(153.93185310477014))


## 6. Summary — Kết luận H0/H1

In [17]:
t1 = stats.ttest_ind(low, high, equal_var=False)
f_stat, p_anova = stats.f_oneway(g_low, g_mod, g_high)
t2 = stats.ttest_ind(trained, untrained, equal_var=False)
chi2, p_chi, dof, _ = stats.chi2_contingency(
    pd.crosstab(df["volume_category"], df["failure_binary"]))

print("="*55)
print("INFERENTIAL STATISTICS SUMMARY")
print("="*55)
results = {
    "t-test Low vs High":      (t1.pvalue,   "hedges_g differs by volume"),
    "ANOVA (3 groups)":        (p_anova,     "at least one group differs"),
    "Chi-sq failure vs vol":   (p_chi,       "failure & volume associated"),
    "t-test Trained vs Untrained": (t2.pvalue, "hedges_g differs by experience"),
}
for test, (p_val, meaning) in results.items():
    decision = "REJECT H0 *" if p_val < ALPHA else "fail to reject H0"
    print(f"  {test}: p={p_val:.4f} -> {decision}")

INFERENTIAL STATISTICS SUMMARY
  t-test Low vs High: p=0.0009 -> REJECT H0 *
  ANOVA (3 groups): p=0.0003 -> REJECT H0 *
  Chi-sq failure vs vol: p=0.4940 -> fail to reject H0
  t-test Trained vs Untrained: p=0.1014 -> fail to reject H0
